<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/03_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 36.1 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [35]:
!git clone https://github.com/yaranoun/ML-Tech.git

Cloning into 'ML-Tech'...
remote: Enumerating objects: 209, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 209 (delta 8), reused 5 (delta 2), pack-reused 190 (from 1)
Receiving objects: 100% (209/209), 162.66 KiB | 1.44 MiB/s, done.
Resolving deltas: 100% (112/112), done.


In [36]:
%cd /content/ML-Tech

/content/ML-Tech


In [53]:
!git pull origin main

remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 29 (delta 14), reused 9 (delta 6), pack-reused 4 (from 1)
Unpacking objects: 100% (29/29), 76.68 KiB | 1.22 MiB/s, done.
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
   b5f6d3b..1f6dd0c  main       -> origin/main
Updating b5f6d3b..1f6dd0c
Fast-forward
 data/processed/passport_index.faiss | Bin 0 -> 61485 bytes
 notebooks/02_embeddings.ipynb       | 390 +++++++++------
 notebooks/03_rag_pipeline.ipynb     | 932 ++++++++++++++++++++----------------
 3 files changed, 758 insertions(+), 564 deletions(-)
 create mode 100644 data/processed/passport_index.faiss


In [56]:
import faiss
index = faiss.read_index("data/processed/passport_index.faiss")

In [60]:
import json

with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

In [61]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-base"
)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [62]:
def retrieve(question, k=3):
    query_embedding = embedding_model.encode(
        ["query: " + question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "document": chunks[idx]["document"],
            "section": chunks[idx]["section"],
            "text": chunks[idx]["text"],
            "url": chunks[idx]["url"]
        })

    return results

In [64]:
results = retrieve(
    "What should I do if my passport was stolen?",
    k=3
)
for result in results:
    print(result["section"], result["score"])

For the individual planning on shipping his passport with another traveler 0.8526734113693237
NB 0.8486536741256714
Stolen Passport 0.8430768847465515


In [65]:
!pip install -q transformers accelerate

In [66]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

llm_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(llm_name)

llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [68]:
question = "What should I do if my passport is stolen?"

results = retrieve(question, k=3)

In [69]:
context = "\n\n".join(
    f"Section: {result['section']}\n{result['text']}"
    for result in results
)

print(context)

Section: For the individual planning on shipping his passport with another traveler
1-   The owner needs to show up in person at the department of press – general security, with the traveler concerned, to convey the pre-mentioned request.
2-   The traveler has to have his airplane ticket in hand to underline the date of his departure, as well as proof of an entry visa and a stable residence in the country of destination
3-   The traveler is held responsible in case of losing the passport, on in case of any illegal use of the latter

Section: NB
The general directorate of General security informs citizens of the necessity to declare a lost/stolen passport even if they don’t intend to apply for a new one.

Section: Stolen Passport
The same above-mentioned procedures are applied for stolen passports, with the exception of an additional document required:
A certified copy of the investigation report done by the ISF internal security forces\


In [70]:
messages = [
    {
        "role": "system",
        "content": """
You are an assistant for Lebanese passport procedures.

Answer using ONLY the official government context provided.

If the answer is not contained in the context, say:
"I could not find this information in the available official sources."

Do not invent procedures, documents, fees, or requirements.
Answer clearly and concisely.
"""
    },
    {
        "role": "user",
        "content": f"""
Official context:

{context}

Question:
{question}
"""
    }
]

In [ ]:
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(llm.device)

with torch.no_grad():
    outputs = llm.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(answer)

In [ ]:
sources = []

for result in results:
    if result["url"] not in sources:
        sources.append(result["url"])

print("\nSources:")
for source in sources:
    print(source)